# Results Analysis & Visualization

This notebook visualizes the final docking results, creating a combined plot containing:
1.  A bar chart comparing Binding Affinity.
2.  A side-table showing the R-group substitutions for each candidate.

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import re
from datetime import datetime

## Helper Functions

`clean_chemical_name`: Formats ligand names for display by replacing underscores with slashes and converting numbers to LaTeX subscripts.

In [2]:
def clean_chemical_name(text):
    if not isinstance(text, str):
        return text
    
    # Replace separator "_" with " / "
    text = text.replace("_", " / ")
    
    # Convert numbers to LaTeX subscripts
    text = re.sub(r"(\d+)", r"$_{\1}$", text)
    
    return text

## Load Data

Load the `final_docking_scores.csv` from the most recent run (or a specific date). 
We select the Top 20 candidates sorted by Affinity (lowest energy is best).

In [3]:
# Determine path (defaulting to today's run, adjust as needed)
run_date = datetime.now().strftime("%Y-%m-%d")
file_path = Path("..") / "data" / "docking" / f"run_{run_date}" / "final_docking_scores.csv"

if not file_path.exists():
    print(f"Warning: {file_path} not found. Trying to find any recent run...")
    docking_dir = Path("..") / "data" / "docking"
    runs = sorted(list(docking_dir.glob("run_*")))
    if runs:
        file_path = runs[-1] / "final_docking_scores.csv"
        print(f"Using: {file_path}")
    else:
        raise FileNotFoundError("No docking results found.")

df = pd.read_csv(file_path)

# Pre-processing
top_n = 20
df_top = df.sort_values(by='Affinity', ascending=True).head(top_n).copy()

# Format names for the plot
df_top['Short_Name'] = df_top['COMB'].apply(clean_chemical_name)

## Visualization

Generate a Bar plot of Binding Affinity with a corresponding data table on the left side showing the specific R-group substitutions.

In [4]:
# Prepare Table Data
split_df = df_top['Short_Name'].str.split(" / ", expand=True)

# Column Alignment (Ensure 4 columns for R1-R4)
target_cols = 4
current_cols = split_df.shape[1]

if current_cols < target_cols:
    for i in range(target_cols - current_cols):
        split_df[current_cols + i] = ""
elif current_cols > target_cols:
    split_df = split_df.iloc[:, :target_cols]

# Style Settings
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams['font.family'] = 'sans-serif'

# Figure Layout
plt.figure(figsize=(8, 6))

# 1. Bar Plot
ax1 = sns.barplot(
    data=df_top, 
    x='Affinity', 
    y='Short_Name', 
    palette='mako', 
    hue='Short_Name', # Fix FutureWarning
    legend=False,
    edgecolor='black'
)

# Clean Axes
ax1.set_ylabel("")          
ax1.set_xlabel("Binding Affinity (kcal/mol)")  
ax1.set_yticklabels([])       
ax1.tick_params(axis='y', length=0) 

# Reference Line for Curcumin (Approximate or Dynamic if in data)
curcumin_val = df[df['Ligand'] == 'Curcumin']['Affinity'].min() if 'Curcumin' in df['Ligand'].values else -7.5
plt.axvline(x=curcumin_val, color='red', linestyle='--', linewidth=2, label=f"Curcumin ({curcumin_val})")

# 2. Table Generation
rows = len(df_top)
table_height = (rows + 1) / rows 

# Add Index if desired or just R-groups
# custom_indices = [f"{i+1}" for i in range(len(split_df))]
# split_df.insert(0, 'Rank', custom_indices)

the_table = plt.table(
    cellText=split_df.values,
    colLabels=["R1", "R2", "R3", "R4"], 
    cellLoc='center',
    loc='left',
    bbox=[-0.45, 0, 0.4, table_height] # [left, bottom, width, height]
)

# Final Polish
for i in ax1.containers:
    ax1.bar_label(i, fmt='%.1f', padding=5, fontsize=10, color='black')

sns.despine(left=True, bottom=True)
plt.legend(loc='lower right') 
plt.title("Top 20 Candidates by Binding Affinity")

plt.show()